# វគ្គ 5 – អ្នកសម្របសម្រួលភ្នាក់ងារច្រើន

បង្ហាញបណ្តាញធ្វើការងារសាមញ្ញសម្រាប់ភ្នាក់ងារពីរ (អ្នកស្រាវជ្រាវ -> អ្នកកែសម្រួល) ដោយប្រើ Foundry Local.


### ការពន្យល់៖ ការដំឡើងអាស្រ័យភាព
ដំឡើង `foundry-local-sdk` និង `openai` ដែលចាំបាច់សម្រាប់ចូលប្រើម៉ូឌែលក្នុងស្រុក និងសម្រាប់បញ្ចប់ការជជែក។ អនុវត្តបានច្រើនដងដោយលទ្ធផលដូចគ្នា។


# សេណារីយ៉ូ
អនុវត្តលំនាំ orchestrator តូចមួយ សម្រាប់ភ្នាក់ងារពីរ៖
- **ភ្នាក់ងារស្រាវជ្រាវ** ប្រមូលចំណុចខ្លីៗ ដែលផ្អែកលើข้อเท็จจริง
- **ភ្នាក់ងារកែសម្រួល** សរសេរឡើងវិញដើម្បីភាពច្បាស់សម្រាប់អ្នកដឹកនាំ

បង្ហាញពីការចងចាំរួមសម្រាប់រាល់ភ្នាក់ងារ, ការបញ្ជូនលំដាប់នៃលទ្ធផលមធ្យម, និងមុខងារប្រព័ន្ធបន្ទាត់សាមញ្ញ។ អាចពង្រីកទៅតួនាទីបន្ថែម (ឧ. Critic, Verifier) ឬសាខាដំណើរការពហុរូប។

**អថេរបរិស្ថាន:**
- `FOUNDRY_LOCAL_ALIAS` - ម៉ូឌែលលំនាំដែលត្រូវប្រើ (លំនាំដើម: phi-4-mini)
- `AGENT_MODEL_PRIMARY` - ម៉ូឌែលសម្រាប់ភ្នាក់ងារចម្បង (ជំនួស ALIAS)
- `AGENT_MODEL_EDITOR` - ម៉ូឌែលសម្រាប់ភ្នាក់ងារកែសម្រួល (លំនាំដើម: ប្រើម៉ូឌែលចម្បង)

**យោង SDK:** https://github.com/microsoft/Foundry-Local/tree/main/sdk/python/foundry_local

**របៀបដែលវាដំណើរការ:**
1. **FoundryLocalManager** ចាប់ផ្តើមសេវាកម្ម Foundry Local ដោយស្វ័យប្រវត្តិ
2. ទាញយក និងផ្ទុកម៉ូឌែលដែលបានបញ្ជាក់ (ឬប្រើកំណែកដែលបានរក្សាទុក)
3. ផ្តល់ច្រកចេញដែលសមស្របជាមួយ OpenAI សម្រាប់អន្តរកម្ម
4. ភ្នាក់ងារ​មួយៗអាចប្រើម៉ូឌែលផ្សេងៗគ្នាសម្រាប់ភារកិច្ចឯកទេស
5. លទ្ធិសាកឡើងវិញដែលបានបញ្ចូលនៅក្នុងប្រព័ន្ធ ដោះស្រាយកំហុសបណ្តោះអាសន្នដោយយុត្តិធម៌

**លក្ខណៈសំខាន់ៗ:**
- ✅ ការស្វែងរកសេវាកម្ម និងការចាប់ផ្តើមដោយស្វ័យប្រវត្តិ
- ✅ គ្រប់គ្រងរយៈពេលជីវិតម៉ូឌែល (ទាញយក, រក្សាទុក, ផ្ទុក)
- ✅ សមស្របជាមួយ OpenAI SDK សម្រាប់ API ដែលស្គាល់
- ✅ ការគាំទ្រម៉ូឌែលច្រើនសម្រាប់ភាពឯកទេសរបស់ភ្នាក់ងារ
- ✅ ការដោះស្រាយកំហុសទាន់ចិត្តជាមួយលទ្ធិសាកឡើងវិញ
- ✅ ការសន្និដ្ឋានក្នុងស្រុក (មិនទាមទារការប្រើប្រាស់ cloud API)


In [16]:
# Install dependencies
!pip install -q foundry-local-sdk openai

### ការពន្យល់៖ ការនាំចូលស្នូល និង ការបញ្ចាក់ប្រភេទ
ណែនាំ dataclasses សម្រាប់ផ្ទុកសាររបស់ភ្នាក់ងារ និង typing hints ដើម្បីភាពច្បាស់លាស់។ នាំចូល Foundry Local manager + OpenAI client សម្រាប់សកម្មភាពបន្ទាប់របស់ភ្នាក់ងារ។


In [17]:
from dataclasses import dataclass, field
from typing import List
import os
from foundry_local import FoundryLocalManager
from openai import OpenAI

### ពន្យល់៖ ការចាប់ផ្ដើមម៉ូឌែល (លំនាំ SDK)
ប្រើ Foundry Local Python SDK សម្រាប់ការគ្រប់គ្រងម៉ូឌែលយ៉ាងរឹងមាំ:
- **FoundryLocalManager(alias)** - ចាប់ផ្ដើមសេវាកម្មដោយស្វ័យប្រវត្តិ និងផ្ទុកម៉ូឌែលដោយ alias
- **get_model_info(alias)** - ដោះស្រាយ alias ទៅជា ID ម៉ូឌែល ជាក់លាក់
- **manager.endpoint** - ផ្តល់ចំណុច endpoint សម្រាប់ client របស់ OpenAI
- **manager.api_key** - ផ្តល់ API key (មិនចាំបាច់សម្រាប់ការប្រើក្នុងលក្ខខណ្ឌក្នុងស្រុក)
- គាំទ្រម៉ូឌែលផ្តាច់មុខសម្រាប់ភ្នាក់ងារផ្សេងៗ (ចម្បង ទល់នឹង អ្នកកែ)
- មានយុទ្ធសាស្ត្រព្យាយាមឡើងវិញ (retry) ជាមួយ exponential backoff សម្រាប់ភាពរឹងមាំ
- ផ្ទៀងផ្ទាត់ការតភ្ជាប់ ដើម្បីធានាថាសេវាកម្មបានត្រៀមរួច

**លំនាំ SDK សំខាន់៖**
```python
manager = FoundryLocalManager(alias)
model_info = manager.get_model_info(alias)
client = OpenAI(base_url=manager.endpoint, api_key=manager.api_key)
```

**ការគ្រប់គ្រងអាយុកាល៖**
- អ្នកគ្រប់គ្រងត្រូវបានផ្ទុកទុកជាសកល ដើម្បីធានាការសម្អាតបានយ៉ាងត្រឹមត្រូវ
- ភ្នាក់ងារនីមួយៗអាចប្រើម៉ូឌែលខុសៗគ្នាសម្រាប់ការប្រកួតពិសេស
- ការស្វែងរកសេវាកម្មដោយស្វ័យប្រវត្តិ និងការដោះស្រាយការតភ្ជាប់
- ព្យាយាមឡើងវិញយ៉ាងសុភាពជាមួយ exponential backoff នៅពេលមានការបរាជ័យ

នេះធានាថាការចាប់ផ្ដើមបានត្រឹមត្រូវ មុនពេលការរៀបចំភ្នាក់ងារចាប់ផ្ដើម។

**យោង៖** https://github.com/microsoft/Foundry-Local/tree/main/sdk/python/foundry_local


In [18]:
import time

# Environment configuration
PRIMARY_ALIAS = os.getenv('AGENT_MODEL_PRIMARY', os.getenv('FOUNDRY_LOCAL_ALIAS', 'phi-4-mini'))
EDITOR_ALIAS = os.getenv('AGENT_MODEL_EDITOR', PRIMARY_ALIAS)

# Store managers globally for proper lifecycle management
primary_manager = None
editor_manager = None

def init_model(alias: str, max_retries: int = 3):
    """Initialize Foundry Local manager with retry logic.
    
    Args:
        alias: Model alias to initialize
        max_retries: Number of retry attempts with exponential backoff
    
    Returns:
        Tuple of (manager, client, model_id, endpoint)
    """
    delay = 2.0
    last_err = None
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"[Init] Starting Foundry Local for '{alias}' (attempt {attempt}/{max_retries})...")
            
            # Initialize manager - this starts the service and loads the model
            manager = FoundryLocalManager(alias)
            
            # Get model info to retrieve the actual model ID
            model_info = manager.get_model_info(alias)
            model_id = model_info.id
            
            # Create OpenAI client with manager's endpoint
            client = OpenAI(
                base_url=manager.endpoint,
                api_key=manager.api_key or 'not-needed'
            )
            
            # Verify the connection with a simple test
            models = client.models.list()
            print(f"[OK] Initialized '{alias}' -> {model_id} at {manager.endpoint}")
            
            return manager, client, model_id, manager.endpoint
            
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Failed to init '{alias}': {e}")
                print(f"[Retry] Waiting {delay:.1f}s before retry...")
                time.sleep(delay)
                delay *= 2
            else:
                print(f"[ERROR] Failed to initialize '{alias}' after {max_retries} attempts")
    
    raise RuntimeError(f"Failed to initialize '{alias}' after {max_retries} attempts: {last_err}")

# Initialize primary model (for researcher)
print(f"\n{'='*80}")
print(f"Initializing Primary Model: {PRIMARY_ALIAS}")
print('='*80)
primary_manager, primary_client, PRIMARY_MODEL_ID, primary_endpoint = init_model(PRIMARY_ALIAS)

# Initialize editor model (may be same as primary)
if EDITOR_ALIAS != PRIMARY_ALIAS:
    print(f"\n{'='*80}")
    print(f"Initializing Editor Model: {EDITOR_ALIAS}")
    print('='*80)
    editor_manager, editor_client, EDITOR_MODEL_ID, editor_endpoint = init_model(EDITOR_ALIAS)
else:
    print(f"\n[Info] Editor using same model as primary")
    editor_manager = primary_manager
    editor_client, EDITOR_MODEL_ID = primary_client, PRIMARY_MODEL_ID
    editor_endpoint = primary_endpoint

print(f"\n{'='*80}")
print(f"[Configuration Summary]")
print('='*80)
print(f"  Primary Agent:")
print(f"    - Alias: {PRIMARY_ALIAS}")
print(f"    - Model: {PRIMARY_MODEL_ID}")
print(f"    - Endpoint: {primary_endpoint}")
print(f"\n  Editor Agent:")
print(f"    - Alias: {EDITOR_ALIAS}")
print(f"    - Model: {EDITOR_MODEL_ID}")
print(f"    - Endpoint: {editor_endpoint}")
print('='*80)



Initializing Primary Model: phi-4-mini
[Init] Starting Foundry Local for 'phi-4-mini' (attempt 1/3)...
[OK] Initialized 'phi-4-mini' -> Phi-4-mini-instruct-cuda-gpu:4 at http://127.0.0.1:59959/v1

Initializing Editor Model: gpt-oss-20b
[Init] Starting Foundry Local for 'gpt-oss-20b' (attempt 1/3)...
[OK] Initialized 'gpt-oss-20b' -> gpt-oss-20b-cuda-gpu:1 at http://127.0.0.1:59959/v1

[Configuration Summary]
  Primary Agent:
    - Alias: phi-4-mini
    - Model: Phi-4-mini-instruct-cuda-gpu:4
    - Endpoint: http://127.0.0.1:59959/v1

  Editor Agent:
    - Alias: gpt-oss-20b
    - Model: gpt-oss-20b-cuda-gpu:1
    - Endpoint: http://127.0.0.1:59959/v1


### ការពន្យល់: Agent & Memory Classes

កំណត់ `AgentMsg` ទម្រង់ស្រាលសម្រាប់ធាតុចុះបញ្ជីអង្គចងចាំ និង `Agent` ដែលរួមបញ្ចូល:

- **System role** - អត្តសញ្ញាណ និងការណែនាំរបស់ Agent
- **Message history** - ថែរក្ស​បរិបទ​នៃ​ការ​សន្ទនា
- **act() method** - បំពេញ​សកម្មភាព​ជាមួយ​ការ​ដោះស្រាយ​កំហុស​យ៉ាង​ត្រឹមត្រូវ

ភ្នាក់ងារ​អាច​ប្រើ​ម៉ូឌែល​ផ្សេងៗ (ម៉ូឌែល​ចម្បង និង ម៉ូឌែល​កែសម្រួល) ហើយថែរក្ស​បរិបទ​ផ្ទាល់ខ្លួន​សម្រាប់រាល់ភ្នាក់ងារ។ លំនាំ​នេះ​អនុញ្ញាត​ឲ្យ៖

- ការរក្សាចងចាំឆ្លងកាត់សកម្មភាព
- ការចាត់តាំងម៉ូឌែលបានយ៉ាងបត់បែនសម្រាប់រាល់ភ្នាក់ងារ
- ការបំបែកកំហុស និងការស្ដារឡើងវិញ
- ការចងជាប់គ្នា និងការរៀបចំប្រតិបត្តិការបានយ៉ាងងាយ


In [19]:
@dataclass
class AgentMsg:
    role: str
    content: str

@dataclass
class Agent:
    name: str
    system: str
    client: OpenAI = None  # Allow per-agent client assignment
    model_id: str = None   # Allow per-agent model
    memory: List[AgentMsg] = field(default_factory=list)

    def _history(self):
        """Return chat history in OpenAI messages format including system + memory."""
        msgs = [{'role': 'system', 'content': self.system}]
        for m in self.memory[-6:]:  # Keep last 6 messages to avoid context overflow
            msgs.append({'role': m.role, 'content': m.content})
        return msgs

    def act(self, prompt: str, temperature: float = 0.4, max_tokens: int = 300):
        """Send a prompt, store user + assistant messages in memory, and return assistant text.
        
        Args:
            prompt: User input/task for the agent
            temperature: Sampling temperature (0.0-1.0)
            max_tokens: Maximum tokens to generate
        
        Returns:
            Assistant response text
        """
        # Use agent-specific client/model or fall back to primary
        client_to_use = self.client or primary_client
        model_to_use = self.model_id or PRIMARY_MODEL_ID
        
        self.memory.append(AgentMsg('user', prompt))
        
        try:
            # Build messages including system prompt and history
            messages = self._history() + [{'role': 'user', 'content': prompt}]
            
            resp = client_to_use.chat.completions.create(
                model=model_to_use,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
            )
            
            # Validate response
            if not resp.choices:
                raise RuntimeError("No completion choices returned")
            
            out = resp.choices[0].message.content or ""
            
            if not out:
                raise RuntimeError("Empty response content")
            
        except Exception as e:
            out = f"[ERROR:{self.name}] {type(e).__name__}: {str(e)}"
            print(f"[Agent Error] {self.name}: {type(e).__name__}: {str(e)}")
        
        self.memory.append(AgentMsg('assistant', out))
        return out

print("[INFO] Agent classes initialized with Foundry SDK support")
print(f"[INFO] Using OpenAI SDK version: {OpenAI.__module__}")


[INFO] Agent classes initialized with Foundry SDK support
[INFO] Using OpenAI SDK version: openai


### ការពិពណ៌នា: លំហូរដំណើរការដែលបានរៀបចំ
បង្កើតភ្នាក់ងារពិសេសចំនួនពីរ៖
- **Researcher**: ប្រើម៉ូឌែលចម្បង និងប្រមូលព័ត៌មានជាការពិត
- **Editor**: អាចប្រើម៉ូឌែលផ្សេង (បើមានការកំណត់), លៃតម្រូវ និងរៀបចំឡើងវិញ

មុខងារ `pipeline`៖
1. Researcher ប្រមូលព័ត៌មានដើម
2. Editor លៃតម្រូវទៅជាលទ្ធផលដែលត្រៀមសម្រាប់អ្នកគ្រប់គ្រង
3. ត្រឡប់ទាំងលទ្ធផលមធ្យម និងលទ្ធផលចុងក្រោយ

រចនាបទនេះអនុញ្ញាតឲ្យមាន៖
- ការបំពាក់ម៉ូឌែលជាពិសេស (ម៉ូឌែលខុសគ្នាសម្រាប់តួនាទីនីមួយៗ)
- ការកែលម្អគុណភាពតាមរយៈដំណើរការជាច្រើនជំហាន
- តាមដានការបម្លែងព័ត៌មានបានងាយ
- ងាយពង្រីកទៅកាន់ភ្នាក់ងារច្រើនទៀត ឬដំណើរការដោយស្របគ្នា


In [ ]:
# Create specialized agents with optional model assignment
researcher = Agent(
    name='Researcher',
    system='You collect concise factual bullet points.',
    client=primary_client,
    model_id=PRIMARY_MODEL_ID
)

editor = Agent(
    name='Editor',
    system='You rewrite content for clarity and an executive, action-focused tone.',
    client=editor_client,
    model_id=EDITOR_MODEL_ID
)

def pipeline(q: str, verbose: bool = True):
    """Execute multi-agent pipeline: Researcher -> Editor.
    
    Args:
        q: User question/task
        verbose: Print intermediate outputs
    
    Returns:
        Dictionary with research, final outputs, and metadata
    """
    if verbose:
        print(f"[Pipeline] Question: {q}\n")
    
    # Stage 1: Research
    if verbose:
        print("[Stage 1: Research]")
    research = researcher.act(q)
    if verbose:
        print(f"Output: {research[:200]}...\n")
    
    # Stage 2: Editorial refinement
    if verbose:
        print("[Stage 2: Editorial Refinement]")
    rewrite = editor.act(
        f"Rewrite professionally with a 1-sentence executive summary first. "
        f"Improve clarity, keep bullet structure if present. Source:\n{research}"
    )
    if verbose:
        print(f"Output: {rewrite[:200]}...\n")
    
    return {
        'question': q,
        'research': research,
        'final': rewrite,
        'models': {
            'researcher': PRIMARY_MODEL_ID,
            'editor': EDITOR_MODEL_ID
        }
    }

# Execute sample pipeline
print("="*80)
result = pipeline('Explain why edge AI matters for compliance and latency.')
print("="*80)
print("\n[FINAL OUTPUT]")
print(result['final'])
print("\n[METADATA]")
print(f"Models used: {result['models']}")
result

[Pipeline] Question: Explain why edge AI matters for compliance and latency.

[Stage 1: Research]
Output: - **Data Sovereignty**: Edge AI allows data to be processed locally, which can help organizations comply with regional data protection regulations by keeping sensitive information within the borders o...

[Stage 2: Editorial Refinement]


### ការពិពណ៌នា៖ ការប្រតិបត្តិប៉ាយប្លាញ និងលទ្ធផល
អនុវត្តប៉ាយប្លាញពហុភ្នាក់ងារលើសំណួរដែលមានប្រធានបទអនុលោម និងភាពយឺត ដើម្បីបង្ហាញ៖
- ការបម្លែងព័ត៌មានច្រើនដំណាក់កាល
- ការជំនាញពិសេសរបស់ភ្នាក់ងារ និងការសហការគ្នា
- ការកែលម្អគុណភាពលទ្ធផលតាមរយៈការកែតម្រូវ
- ភាពអាចតាមដានបាន (លទ្ធផលចន្លោះ និងចុងក្រោយទាំងពីរត្រូវបានរក្សា)

**រចនាសម្ព័ន្ធលទ្ធផល:**
- `question` - សំណួរដើម​របស់អ្នកប្រើ
- `research` - លទ្ធផលស្រាវជ្រាវដើម (ចំណុចពិត)
- `final` - សេចក្តីសង្ខេបចុងក្រោយដែលបានកែតម្រូវ
- `models` - តើម៉ូដែលណាខ្លះដែលបានប្រើសម្រាប់រាល់ដំណាក់កាល

**គំនិតផ្នែកបន្ថែម:**
1. បន្ថែមភ្នាក់ងារវិភាគសម្រាប់ពិនិត្យគុណភាព
2. អនុវត្តភ្នាក់ងារស្រាវជ្រាវដំណើរការដោយស្របក្នុងពេលដដែលសម្រាប់ជំពូកផ្សេងៗ
3. បន្ថែមភ្នាក់ងារ Verifier សម្រាប់ពិនិត្យភាពត្រឹមត្រូវ
4. ប្រើម៉ូដែលខុសៗគ្នាសម្រាប់កម្រិតស្មុគស្មាញផ្សេងៗ
5. អនុវត្តលំនុំតបត្រឡប់សម្រាប់កែលម្អដោយជាបន្ត


### កម្រិតខ្ពស់: ការកំណត់ផ្ទាល់ខ្លួនសម្រាប់ភ្នាក់ងារ

សាកល្បងប្ដូរប្រព្រឹត្តិការណ៍របស់ភ្នាក់ងារ ដោយកែប្រែអថេរបរិស្ថាន មុនពេលធ្វើការរត់កោសិកាចាប់ផ្តើម:

**ម៉ូឌែលដែលមានស្រាប់:**
- ប្រើ `foundry model ls` នៅក្នុងផ្ទាំងបញ្ជា ដើម្បីមើលម៉ូឌែលទាំងអស់ដែលមាន
- ឧទាហរណ៍: phi-4-mini, phi-3.5-mini, qwen2.5-7b, llama-3.2-3b, ជាដើម.


In [ ]:
# Example: Use different models for different agents
# Uncomment and modify as needed:

# import os
# os.environ['AGENT_MODEL_PRIMARY'] = 'phi-4-mini'      # Fast, good for research
# os.environ['AGENT_MODEL_EDITOR'] = 'qwen2.5-7b'       # Higher quality for editing

# Then restart the kernel and re-run all cells

# Test with different questions
test_questions = [
    "What are 3 key benefits of using small language models?",
    "How does RAG improve AI accuracy?",
    "Why is local inference important for privacy?"
]

print("Testing pipeline with multiple questions:\n")
for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*80}")
    print(f"Question {i}: {q}")
    print('='*80)
    r = pipeline(q, verbose=False)
    print(f"\n[FINAL]: {r['final'][:300]}...")
    print(f"[Models]: Researcher={r['models']['researcher']}, Editor={r['models']['editor']}")


Testing pipeline with multiple questions:


Question 1: What are 3 key benefits of using small language models?

[FINAL]: <|channel|>analysis<|message|>The user wants a rewrite of the entire block of text. The rewrite should be professional, include a one-sentence executive summary first, improve clarity, keep bullet structure if present. The user has provided a large amount of text. The user wants a rewrite of that te...
[Models]: Researcher=Phi-4-mini-instruct-cuda-gpu:4, Editor=gpt-oss-20b-cuda-gpu:1

Question 2: How does RAG improve AI accuracy?

[FINAL]: <|channel|>final<|message|>**RAG (Retrieval‑Augmented Generation) empowers AI to produce highly accurate, contextually relevant responses by combining a retrieval system with a large language model (LLM).**<|return|>...
[Models]: Researcher=Phi-4-mini-instruct-cuda-gpu:4, Editor=gpt-oss-20b-cuda-gpu:1

Question 3: Why is local inference important for privacy?

[FINAL]: <|channel|>final<|message|>**Local inference—processing data d

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារ​នេះ​បាន​បកប្រែ​ដោយ​ប្រើ​សេវាកម្ម​បកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះ​យើង​ខិតខំ​សម្រាប់ភាពត្រឹមត្រូវ ក៏​សូម​ជ្រាប​ថា ការបកប្រែ​ដោយស្វ័យប្រវត្តិ​អាច​មានកំហុស ឬ​មិន​ត្រឹម​ត្រូវ។ ឯកសារ​ដើម​ក្នុង​ភាសាដើម​របស់វា​គួរត្រូវបាន​ចាត់ទុកថាជា​ប្រភព​ដែល​មាន​សុពលភាព។ សម្រាប់ព័ត៌មាន​សំខាន់ៗ យើង​សូមណែនាំ​ឲ្យ​ប្រើ​ការ​បកប្រែ​ដោយ​អ្នកជំនាញ​មនុស្ស។ យើង​មិន​ទទួល​ខុសត្រូវ​ចំពោះ​ការ​យល់ច្រឡំ ឬ​ការ​បកស្រាយ​ខុស​ដែល​កើតឡើង​ពី​ការ​ប្រើប្រាស់​ការ​បកប្រែ​នេះ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
